### Configure PyTorch CUDA memory allocation settings to prevent fragmentation

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

### Mount Google Drive for persistent storage

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


### Check GPU availability and display GPU information

In [ ]:
gpu_info = !nvidia-smi
gpu_info = "\n".join(gpu_info)
if gpu_info.find("failed") >= 0:
    print("Not connected to a GPU")
else:
    print(gpu_info)


Thu Mar 12 16:59:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Install required Python packages (datasets, transformers, torchaudio, jiwer, accelerate, evaluate)

In [ ]:
%%capture
!pip install datasets==3.6.0
!pip install transformers==4.57.1
!pip install torchaudio
!pip install jiwer
!pip install accelerate -U
!pip install evaluate


### Authenticate with Hugging Face Hub

In [ ]:
from huggingface_hub import notebook_login
notebook_login()


### Load the Dhivehi (Maldivian) Common Voice dataset (train+validation and test splits)

In [ ]:
from datasets import load_dataset

common_voice_train = load_dataset("fsicoli/common_voice_22_0", "dv", split="train+validation", trust_remote_code=True)
common_voice_test = load_dataset("fsicoli/common_voice_22_0", "dv", split="test", trust_remote_code=True)

### Remove unused metadata columns (accent, age, client_id, etc.) from train and test sets and display random samples

In [ ]:
common_voice_train = common_voice_train.remove_columns(
    ["accent", "age", "client_id", "down_votes", "gender", "locale", "segment", "up_votes"]
)
common_voice_test = common_voice_test.remove_columns(
    ["accent", "age", "client_id", "down_votes", "gender", "locale", "segment", "up_votes"]
)

from datasets import ClassLabel
import random
import pandas as pd
from IPython.display import display, HTML

def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset) - 1)
        while pick in picks:
            pick = random.randint(0, len(dataset) - 1)
        picks.append(pick)
    df = pd.DataFrame(dataset[picks])
    display(HTML(df.to_html()))

# Select the 'train' split before calling show_random_elements
show_random_elements(common_voice_train.remove_columns(["path", "audio"]), num_examples=10)


,sentence,variant
0,ކާށިތެލަކީ މާ މޮޅު އެއްޗެއްތަ؟,
1,ގދ.ގައްދޫ ފުޓްސަލް ދަނޑު މަރާމާތުކުރުމާއި ބެހޭ,
2,ކޮން އިރަކުން ކިޔެވުން ފެށެނީ؟,
3,ވަޒީފާގެ ފުރުސަތު - އެޑިޔުކޭޝަން އޮފިސަރ,
4,ޝަބާބްގެ އަޑު ބެދުން,
5,ފައިސާ ކަނޑާ ގޮނޑީގައި އިނީ އެހެން ކުއްޖެއް,
6,ކޮންމެކަމަކާމެދުގައި ޖަދަލުކޮށް ޒުވާބުކުރުމަކީ ވިސްނުންކޮށި މީހުންކޮށްއުޅޭ ތާކުންތާކުނުޖެހޭ ވަރަށް ދެރަކަމެއް,
7,މި ނުރައްކަލުގެ ތެރެއަށް ވަންނާން ޖެހޭނެ,
8,އެއަރ ކޮންޑިޝަނަރ ހޯދުމަށް,
9,އެއީ ކަންތައް ކުރެވުނީ ރަނގަޅަށް ކަމުގެ އިހުސާސް ކުރެވުނީމާ ލިބުނު ހިތްހަމަ ޖެހުމުގެ އަސަރު,


### Define and apply text cleaning: remove punctuation, special characters, and filter out Arabic script rows

In [ ]:
import re

chars_to_remove_regex = r"""[\,\?\.\!\-\;\:\"\“\%\‘\”\�\'\»\«\–\’]"""

def remove_special_characters(batch):
    batch["sentence"] = re.sub(chars_to_remove_regex, "", batch["sentence"]).lower().strip()
    return batch

common_voice_train = common_voice_train.map(remove_special_characters)
common_voice_test  = common_voice_test.map(remove_special_characters)

arabic_pattern = re.compile(r"[\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF\uFB50-\uFDFF\uFE70-\uFEFF]")

def remove_arabic_rows(batch):
    return not bool(arabic_pattern.search(batch["sentence"]))

common_voice_train = common_voice_train.filter(remove_arabic_rows)
common_voice_test  = common_voice_test.filter(remove_arabic_rows)

Map:   0%|          | 0/4897 [00:00<?, ? examples/s]

Map:   0%|          | 0/2222 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4897 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2222 [00:00<?, ? examples/s]

### Extract the unique character vocabulary from train and test sets

In [ ]:
def extract_all_chars(batch):
    all_text = " ".join(batch["sentence"])
    vocab = list(set(all_text))
    return {"vocab": [vocab], "all_text": [all_text]}

vocab_train = common_voice_train.map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=common_voice_train.column_names,
)
vocab_test = common_voice_test.map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=common_voice_test.column_names,
)

vocab_list = list(set(vocab_train["vocab"][0]) | set(vocab_test["vocab"][0]))
vocab_dict = {v: k for k, v in enumerate(sorted(vocab_list))}
print(vocab_dict)

Map:   0%|          | 0/4600 [00:00<?, ? examples/s]

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

{' ': 0, 'ހ': 1, 'ށ': 2, 'ނ': 3, 'ރ': 4, 'ބ': 5, 'ޅ': 6, 'ކ': 7, 'އ': 8, 'ވ': 9, 'މ': 10, 'ފ': 11, 'ދ': 12, 'ތ': 13, 'ލ': 14, 'ގ': 15, 'ޏ': 16, 'ސ': 17, 'ޑ': 18, 'ޒ': 19, 'ޓ': 20, 'ޔ': 21, 'ޕ': 22, 'ޖ': 23, 'ޗ': 24, 'ޘ': 25, 'ޙ': 26, 'ޚ': 27, 'ޛ': 28, 'ޝ': 29, 'ޞ': 30, 'ޟ': 31, 'ޠ': 32, 'ޡ': 33, 'ޢ': 34, 'ޣ': 35, 'ޤ': 36, 'ޥ': 37, 'ަ': 38, 'ާ': 39, 'ި': 40, 'ީ': 41, 'ު': 42, 'ޫ': 43, 'ެ': 44, 'ޭ': 45, 'ޮ': 46, 'ޯ': 47, 'ް': 48}


### Finalize the vocabulary: replace space with pipe delimiter, add [UNK] and [PAD] tokens, and save vocab.json

In [ ]:
# use "|" as word delimiter instead of space
vocab_dict["|"] = vocab_dict[" "]
del vocab_dict[" "]

# add special tokens
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

print("Vocab size:", len(vocab_dict))

import json
with open("vocab.json", "w", encoding="utf-8") as vocab_file:
    json.dump(vocab_dict, vocab_file, ensure_ascii=False)


Vocab size: 51


### Initialize the CTC tokenizer, SeamlessM4T feature extractor, and Wav2Vec2-BERT processor

In [ ]:
from transformers import Wav2Vec2CTCTokenizer
from transformers import SeamlessM4TFeatureExtractor
from transformers import Wav2Vec2BertProcessor

tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(
    "./",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

feature_extractor = SeamlessM4TFeatureExtractor(
    feature_size=80,
    num_mel_bins=80,
    sampling_rate=16000,
    padding_value=0.0,
)

processor = Wav2Vec2BertProcessor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)


### Resample all audio to 16kHz and preview a random training sample

In [ ]:
from datasets import Audio

common_voice_train = common_voice_train.cast_column("audio", Audio(sampling_rate=16_000))
common_voice_test = common_voice_test.cast_column("audio", Audio(sampling_rate=16_000))

print(common_voice_train[0]["audio"])

import IPython.display as ipd
import numpy as np
import random

rand_int = random.randint(0, len(common_voice_train) - 1)
print("Target text:", common_voice_train[rand_int]["sentence"])
print("Input array shape:", common_voice_train[rand_int]["audio"]["array"].shape)
print("Sampling rate:", common_voice_train[rand_int]["audio"]["sampling_rate"])

ipd.Audio(
    data=common_voice_train[rand_int]["audio"]["array"],
    autoplay=True,
    rate=16000,
)


{'path': '/root/.cache/huggingface/datasets/downloads/extracted/e24cb6e0896730e3c68f4266eadc6dbf6150fbe071ba9968bf13481404180b10/dv_train_0/common_voice_dv_18580646.mp3', 'array': array([1.45519152e-11, 1.01863407e-10, 1.30967237e-10, ...,
       7.25846985e-06, 1.29183172e-06, 5.80571304e-06]), 'sampling_rate': 16000}
Target text: ފަރުނީޗަރ ގަތުމަށް
Input array shape: (43200,)
Sampling rate: 16000


### Define the dataset preparation function: extract audio features and encode text labels, then apply to train and test sets

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
    ).input_features[0]
    batch["input_length"] = len(batch["input_features"])
    batch["labels"] = processor(text=batch["sentence"]).input_ids
    return batch

common_voice_train = common_voice_train.map(
    prepare_dataset,
    remove_columns=common_voice_train.column_names,
)
common_voice_test = common_voice_test.map(
    prepare_dataset,
    remove_columns=common_voice_test.column_names,
)


Map:   0%|          | 0/4600 [00:00<?, ? examples/s]

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

### Define a custom data collator that pads input features and labels for CTC training

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Union
import evaluate

from transformers import Wav2Vec2BertProcessor

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2BertProcessor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )
        labels_batch = self.processor.pad(
            labels=label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch["attention_mask"].ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

wer_metric = evaluate.load("wer")


### Define the evaluation metrics function computing WER and CER using greedy decoding

In [ ]:
import numpy as np

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    label_ids = pred.label_ids

    pred_ids = np.argmax(pred_logits, axis=-1)
    pred_str = processor.batch_decode(pred_ids)

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer, "cer": cer}

### Clear GPU memory cache before loading the model

In [ ]:
torch.cuda.empty_cache()

### Load the pre-trained W2V-BERT 2.0 model with an adapter and CTC head for fine-tuning

In [ ]:
from transformers import Wav2Vec2BertForCTC

sinhala_checkpoint_dir = "/content/drive/MyDrive/final_model35"  # <-- wherever your Sinhala ./final_model is

model = Wav2Vec2BertForCTC.from_pretrained(
  sinhala_checkpoint_dir,              # <-- CHANGED (was facebook/w2v-bert-2.0)
  add_adapter=True,
  pad_token_id=processor.tokenizer.pad_token_id,
  vocab_size=len(processor.tokenizer), # keep Dhivehi vocab size here
  ignore_mismatched_sizes=True,        # <-- IMPORTANT for Sinhala->Dhivehi
)


Some weights of Wav2Vec2BertForCTC were not initialized from the model checkpoint at /content/drive/MyDrive/final_model35 and are newly initialized because the shapes did not match:
- lm_head.bias: found shape torch.Size([80]) in the checkpoint and torch.Size([53]) in the model instantiated
- lm_head.weight: found shape torch.Size([80, 1024]) in the checkpoint and torch.Size([53, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Configure training hyperparameters: batch size, learning rate, evaluation strategy, checkpointing, and early stopping

In [ ]:
from transformers import TrainingArguments, Trainer, Wav2Vec2BertForCTC, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="./outputs",
    group_by_length=True,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    gradient_checkpointing=True,
    fp16=True,

    eval_strategy="steps",     # FIXED
    eval_steps=300,

    logging_strategy="steps",
    logging_steps=300,

    save_strategy="steps",
    save_steps=2700,

    learning_rate=5e-5,
    warmup_steps=500,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_wer",   # FIXED
    greater_is_better=False,

    push_to_hub=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=common_voice_train,
    eval_dataset=common_voice_test,
    tokenizer=processor
)


/tmp/ipykernel_16508/723348825.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


### Start the fine-tuning training run

In [21]:
trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 52, 'bos_token_id': 51}.


Step,Training Loss,Validation Loss,Wer,Cer
300,2355.587500,227.804337,0.619581,0.110755
600,663.786700,168.574844,0.500371,0.083642
900,548.268900,155.843887,0.466300,0.073501
1200,479.351000,138.742844,0.443438,0.068830


Step,Training Loss,Validation Loss,Wer,Cer
300,2355.587500,227.804337,0.619581,0.110755
600,663.786700,168.574844,0.500371,0.083642
900,548.268900,155.843887,0.466300,0.073501
1200,479.351000,138.742844,0.443438,0.068830


TrainOutput(global_step=1440, training_loss=910.2059895833333, metrics={'train_runtime': 8857.4831, 'train_samples_per_second': 5.193, 'train_steps_per_second': 0.163, 'total_flos': 7.110251700446906e+18, 'train_loss': 910.2059895833333, 'epoch': 10.0})

### Run greedy CTC inference on the test set and compute WER/CER

In [24]:
from transformers import AutoModelForCTC, Wav2Vec2BertProcessor
import torch
import numpy as np


all_pred = []
all_refs = []

checkpoint_dir = "./outputs/checkpoint-1440"

model = AutoModelForCTC.from_pretrained(checkpoint_dir).to("cuda")
processor = Wav2Vec2BertProcessor.from_pretrained(checkpoint_dir)

for ex in common_voice_test:
    input_feats = torch.tensor(ex["input_features"]).unsqueeze(0).to(model.device)

    with torch.no_grad():
        logits = model(input_feats).logits

    pred_ids = torch.argmax(logits, dim=-1)
    pred_text = processor.batch_decode(pred_ids)[0]
    all_pred.append(pred_text)

    ref_text = processor.decode(ex["labels"], group_tokens=False).lower()
    all_refs.append(ref_text)

wer = wer_metric.compute(predictions=all_pred, references=all_refs)
cer = cer_metric.compute(predictions=all_pred, references=all_refs)

print(f"WER (ASR only, greedy): {wer:.4f}")
print(f"CER (ASR only, greedy): {cer:.4f}")

for idx in range(3):
    print("\n--- Example", idx, "---")
    print("REF :", all_refs[idx])
    print("PRED:", all_pred[idx])

WER (ASR only, greedy): 0.4355
CER (ASR only, greedy): 0.0669

--- Example 0 ---
REF : ޕިކަޕް މަރާމާތުކޮށްދޭނެ ފަރާތެއް ހޯދުން
PRED: ޕިކަށް މަރާމާތުކޮށްދޭނެ ފަރާތެއް ހޯދުން

--- Example 1 ---
REF : އަންޖޫދާ މަޑުމަޑުން އެހެން ބުނެލުމުން ރިޝްކާ ވެސް އިތުރު ސުވާލެއް ނުކުރޭ
PRED: އަންޖޫދާ މަޑުމަޑުން އެހެން ބުނެލުމުން ރިޝްކާވެސް އިތުރު ސުވާލެއް ނުކުރޭ

--- Example 2 ---
REF : އެސްފިޔަތަކުން ވަނީ ނިދި ގެއްލިފަ
PRED: އެސްފިޔަތަކުން ވަނީ ނިދިގެއްލިފަ
